# Text Description Accuracy vs. Semantic Distance Preservation

This notebook evaluates whether LLM-generated category descriptions are accurate enough to classify texts and whether that accuracy is related to how well the descriptions preserve the semantic distances observed in the original category text distributions.

## Workflow implemented

1. Read a Google Drive JSON file containing categories and texts.
2. Split each category into 80% training and 20% testing records.
3. Generate one description per category from the training split using **Gemini 3.1 Flash Lite**.
4. Classify randomized batches of texts with the generated descriptions.
5. Evaluate global and category-level performance with tables and charts.
6. Compute semantic-distance preservation with embeddings by comparing text centroid distances to description distances.
7. Give an absolute correlation answer: whether description accuracy is correlated with semantic-distance preservation.
8. Optionally run an ablation using **Gemini 3.5 Flash**, generating all category descriptions in one request.

> Intermediate LLM responses, embeddings, predictions, and metrics are cached by experiment id. Existing experiment folders are never deleted or overwritten unless `REUSE_CACHE=False` and you provide a new `EXPERIMENT_ID`.


## 0. Runtime setup

Run this cell once per environment. In Colab, install packages first, then restart only if your environment requests it.


In [ ]:
# %pip install -q google-genai pandas numpy scikit-learn matplotlib seaborn scipy tqdm pydantic pyarrow


## 1. Imports and typed data contracts

The notebook uses explicit `TypedDict` contracts so future dataset-building notebooks can reuse the same JSON shape and generated artifacts.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import re
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Literal, NotRequired, TypedDict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial.distance import cosine
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

try:
    from google import genai
    from google.genai import types as genai_types
except Exception as exc:
    genai = None
    genai_types = None
    print("google-genai is not importable yet. Run the install cell, then re-run imports.", exc)

sns.set_theme(style="whitegrid")


class CategoryInput(TypedDict):
    """One labeled group of raw texts in the source JSON file."""

    category: str
    texts: list[str]
    metadata: NotRequired[dict[str, Any]]


class DatasetInput(TypedDict):
    """Preferred source JSON format."""

    categories: list[CategoryInput]
    dataset_name: NotRequired[str]
    metadata: NotRequired[dict[str, Any]]


class DescriptionRecord(TypedDict):
    """Cached LLM-generated category description."""

    category: str
    description: str
    model: str
    prompt_hash: str
    generated_at_utc: str
    raw_response: dict[str, Any]


class PredictionRecord(TypedDict):
    """Cached classification prediction for one text."""

    text_id: str
    split: Literal["train", "test"]
    true_category: str
    predicted_category: str
    confidence: float | None
    rationale: str
    model: str
    batch_id: str


@dataclass(frozen=True)
class ExperimentPaths:
    root: Path
    raw: Path
    splits: Path
    descriptions: Path
    predictions: Path
    embeddings: Path
    metrics: Path
    figures: Path


## 2. Configuration

Set `GOOGLE_DRIVE_JSON_PATH` to a mounted Google Drive path, for example `/content/drive/MyDrive/my_dataset/categories.json`.

If running outside Colab, use any local path. API credentials are read from `GOOGLE_API_KEY`.


In [ ]:
# --- User input: update this path before running the pipeline. ---
GOOGLE_DRIVE_JSON_PATH = "/content/drive/MyDrive/path/to/category_texts.json"

# Optional Colab mount. Set to True when running in Google Colab.
MOUNT_GOOGLE_DRIVE = False

# Experiment identity and cache behavior.
# Use a new id for a new experiment. Existing experiments are not deleted.
EXPERIMENT_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_text_description_eval")
REUSE_CACHE = True

# Reproducibility and split behavior.
RANDOM_SEED = 42
TRAIN_SIZE = 0.80
EVAL_SPLIT: Literal["train", "test"] = "test"  # choose "train" to reproduce the prompt-calibration classification loop exactly
CLASSIFICATION_BATCH_SIZE = 20
MAX_TEXTS_PER_CATEGORY_FOR_DESCRIPTION = 200
MAX_CHARS_PER_TEXT_IN_PROMPTS = 700

# Gemini model configuration. Keep these strings configurable because model availability can vary by account/region.
DESCRIPTION_MODEL = "gemini-3.1-flash-lite"
CLASSIFICATION_MODEL = "gemini-3.1-flash-lite"
ABLATION_DESCRIPTION_MODEL = "gemini-3.5-flash"
EMBEDDING_MODEL = "gemini-embedding-001"

# Safety valve for quick smoke tests; set None for full data.
MAX_TEXTS_PER_CATEGORY: int | None = None

# Optional ablation: all descriptions in a single request with Gemini 3.5 Flash.
RUN_GEMINI_35_SINGLE_REQUEST_ABLATION = False

# Output directory. On Colab, this can also point into Drive to persist all caches.
OUTPUT_ROOT = Path("text-classification/experiments") / EXPERIMENT_ID


## 3. Helpers: mounting, paths, cache-safe writes, and API client

The cache helpers write files atomically and refuse to overwrite existing files unless a caller explicitly chooses a new experiment directory. This prevents accidental quota-heavy re-runs and preserves prior experiments.


In [ ]:
def maybe_mount_drive() -> None:
    if not MOUNT_GOOGLE_DRIVE:
        return
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        raise RuntimeError("Could not mount Google Drive. Are you running in Colab?") from exc


def build_paths(root: Path) -> ExperimentPaths:
    paths = ExperimentPaths(
        root=root,
        raw=root / "raw",
        splits=root / "splits",
        descriptions=root / "descriptions",
        predictions=root / "predictions",
        embeddings=root / "embeddings",
        metrics=root / "metrics",
        figures=root / "figures",
    )
    for path in asdict(paths).values():
        Path(path).mkdir(parents=True, exist_ok=True)
    return paths


def stable_hash(value: Any) -> str:
    payload = json.dumps(value, ensure_ascii=False, sort_keys=True).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:16]


def read_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json_once(path: Path, payload: Any) -> None:
    """Write JSON without clobbering existing cache files."""
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite existing cache: {path}")
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    tmp.replace(path)


def load_or_create_json(path: Path, creator, *, reuse_cache: bool = REUSE_CACHE) -> Any:
    if reuse_cache and path.exists():
        return read_json(path)
    if path.exists() and not reuse_cache:
        raise FileExistsError(
            f"Cache exists and REUSE_CACHE=False: {path}. Choose a new EXPERIMENT_ID to preserve past experiments."
        )
    payload = creator()
    write_json_once(path, payload)
    return payload


def get_genai_client():
    if genai is None:
        raise ImportError("google-genai is not installed. Run the install cell first.")
    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise EnvironmentError("Set GOOGLE_API_KEY before calling Gemini models.")
    return genai.Client(api_key=api_key)


def extract_text_from_response(response: Any) -> str:
    text = getattr(response, "text", None)
    if text:
        return text
    return str(response)


def response_to_jsonable(response: Any) -> dict[str, Any]:
    if hasattr(response, "model_dump"):
        return response.model_dump(mode="json")
    if hasattr(response, "to_json_dict"):
        return response.to_json_dict()
    return {"text": extract_text_from_response(response)}


def gemini_generate_json(client, *, model: str, prompt: str, temperature: float = 0.2, max_retries: int = 5) -> tuple[Any, Any]:
    """Call Gemini and parse a JSON response, retrying transient failures."""
    last_error = None
    for attempt in range(max_retries):
        try:
            config = None
            if genai_types is not None:
                config = genai_types.GenerateContentConfig(
                    temperature=temperature,
                    response_mime_type="application/json",
                )
            response = client.models.generate_content(model=model, contents=prompt, config=config)
            text = extract_text_from_response(response)
            return json.loads(text), response
        except Exception as exc:
            last_error = exc
            sleep_s = min(2 ** attempt, 30) + random.random()
            print(f"Gemini call failed on attempt {attempt + 1}/{max_retries}: {exc}. Sleeping {sleep_s:.1f}s")
            time.sleep(sleep_s)
    raise RuntimeError("Gemini call failed after retries") from last_error


maybe_mount_drive()
paths = build_paths(OUTPUT_ROOT)
print(f"Experiment directory: {paths.root}")


## 4. Load and validate the category JSON

### Expected JSON

```json
{
  "dataset_name": "example_dataset",
  "categories": [
    {"category": "billing", "texts": ["How do I update my card?", "..."]},
    {"category": "technical_support", "texts": ["The app crashes", "..."]}
  ]
}
```

A shorthand mapping is also accepted: `{"billing": ["..."], "technical_support": ["..."]}`.


In [ ]:
def normalize_dataset(raw: Any) -> DatasetInput:
    if isinstance(raw, dict) and "categories" in raw:
        categories = raw["categories"]
        dataset_name = raw.get("dataset_name", "unnamed_dataset")
        metadata = raw.get("metadata", {})
    elif isinstance(raw, dict):
        categories = [{"category": str(k), "texts": v} for k, v in raw.items()]
        dataset_name = "mapping_dataset"
        metadata = {}
    else:
        raise ValueError("Dataset must be either {'categories': [...]} or a mapping of category -> texts.")

    normalized_categories: list[CategoryInput] = []
    seen = set()
    for item in categories:
        category = str(item["category"]).strip()
        if not category:
            raise ValueError("Category names cannot be blank.")
        if category in seen:
            raise ValueError(f"Duplicate category name: {category}")
        seen.add(category)
        texts = [str(t).strip() for t in item["texts"] if str(t).strip()]
        if MAX_TEXTS_PER_CATEGORY is not None:
            texts = texts[:MAX_TEXTS_PER_CATEGORY]
        if len(texts) < 2:
            raise ValueError(f"Category {category!r} needs at least 2 non-empty texts for an 80/20 split.")
        normalized_categories.append({"category": category, "texts": texts, "metadata": item.get("metadata", {})})

    if len(normalized_categories) < 2:
        raise ValueError("At least two categories are required for classification and distance preservation analysis.")
    return {"dataset_name": dataset_name, "metadata": metadata, "categories": normalized_categories}


def load_dataset_from_drive(path_str: str) -> DatasetInput:
    source_path = Path(path_str)
    if not source_path.exists():
        raise FileNotFoundError(f"Dataset JSON not found: {source_path}")
    raw = read_json(source_path)
    dataset = normalize_dataset(raw)
    dataset_fingerprint = stable_hash(dataset)
    raw_cache = paths.raw / f"dataset_{dataset_fingerprint}.json"
    if not raw_cache.exists():
        write_json_once(raw_cache, dataset)
    print(f"Loaded {len(dataset['categories'])} categories from {source_path}")
    print(f"Dataset cache: {raw_cache}")
    return dataset


dataset = load_dataset_from_drive(GOOGLE_DRIVE_JSON_PATH)
pd.DataFrame(
    [{"category": c["category"], "n_texts": len(c["texts"])} for c in dataset["categories"]]
).sort_values("category")


## 5. Stratified per-category 80/20 split

Each category is split independently to ensure every label contributes training and testing examples. For tiny categories, at least one test example is kept.


In [ ]:
def make_splits(dataset: DatasetInput) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    rng = random.Random(RANDOM_SEED)
    for category_item in dataset["categories"]:
        category = category_item["category"]
        texts = list(category_item["texts"])
        rng.shuffle(texts)
        n_total = len(texts)
        n_test = max(1, int(round(n_total * (1 - TRAIN_SIZE))))
        n_train = n_total - n_test
        if n_train < 1:
            raise ValueError(f"Category {category!r} cannot produce at least one train and one test item.")
        for split, split_texts in [("train", texts[:n_train]), ("test", texts[n_train:])]:
            for idx, text in enumerate(split_texts):
                text_id = stable_hash({"category": category, "split": split, "idx": idx, "text": text})
                rows.append({"text_id": text_id, "category": category, "split": split, "text": text})
    return pd.DataFrame(rows)


splits_path = paths.splits / "splits.parquet"
if REUSE_CACHE and splits_path.exists():
    df = pd.read_parquet(splits_path)
else:
    if splits_path.exists():
        raise FileExistsError("Split cache exists. Use a new EXPERIMENT_ID instead of overwriting it.")
    df = make_splits(dataset)
    df.to_parquet(splits_path, index=False)

split_summary = df.groupby(["category", "split"]).size().unstack(fill_value=0)
display(split_summary)


## 6. Generate iterative category descriptions with Gemini 3.1 Flash Lite

The generator makes one cached request per category. It uses only training examples and asks for operational descriptions that can support classification.


In [ ]:
def truncate_text(text: str, max_chars: int = MAX_CHARS_PER_TEXT_IN_PROMPTS) -> str:
    return text if len(text) <= max_chars else text[: max_chars - 1] + "…"


def description_prompt(category: str, examples: list[str], all_categories: list[str]) -> str:
    numbered_examples = "\n".join(f"{i+1}. {truncate_text(t)}" for i, t in enumerate(examples))
    return f"""
You are building category descriptions for a text classifier.

Target category: {category}
All candidate categories: {json.dumps(all_categories, ensure_ascii=False)}

Training examples for the target category:
{numbered_examples}

Write a precise, reusable description of the target category. Include semantic boundaries, inclusion criteria, exclusion criteria, and common edge cases. Avoid mentioning that the description came from examples.

Return JSON exactly in this schema:
{{
  "category": "{category}",
  "description": "..."
}}
""".strip()


def generate_description_for_category(client, category: str, examples: list[str], all_categories: list[str]) -> DescriptionRecord:
    sampled = examples[:]
    random.Random(RANDOM_SEED).shuffle(sampled)
    sampled = sampled[:MAX_TEXTS_PER_CATEGORY_FOR_DESCRIPTION]
    prompt = description_prompt(category, sampled, all_categories)
    parsed, response = gemini_generate_json(client, model=DESCRIPTION_MODEL, prompt=prompt, temperature=0.2)
    return {
        "category": category,
        "description": str(parsed["description"]).strip(),
        "model": DESCRIPTION_MODEL,
        "prompt_hash": stable_hash(prompt),
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "raw_response": response_to_jsonable(response),
    }


def get_descriptions(client=None) -> dict[str, DescriptionRecord]:
    client = client or get_genai_client()
    all_categories = sorted(df["category"].unique().tolist())
    records: dict[str, DescriptionRecord] = {}
    for category in tqdm(all_categories, desc="Generating descriptions"):
        cache_path = paths.descriptions / f"{stable_hash({'category': category, 'model': DESCRIPTION_MODEL})}.json"
        train_examples = df[(df.category == category) & (df.split == "train")]["text"].tolist()
        record = load_or_create_json(
            cache_path,
            lambda category=category, train_examples=train_examples: generate_description_for_category(
                client, category, train_examples, all_categories
            ),
        )
        records[category] = record
    return records


descriptions = get_descriptions()
descriptions_df = pd.DataFrame([{"category": k, "description": v["description"]} for k, v in descriptions.items()])
display(descriptions_df)


## 7. Classify randomized batches of texts

Batches are randomized and capped at 20 texts per request. The prompt receives the frozen category descriptions and must return one prediction per `text_id`.


In [ ]:
def classification_prompt(batch: pd.DataFrame, descriptions: dict[str, DescriptionRecord]) -> str:
    description_block = json.dumps(
        [{"category": c, "description": rec["description"]} for c, rec in sorted(descriptions.items())],
        ensure_ascii=False,
        indent=2,
    )
    text_block = json.dumps(
        [{"text_id": row.text_id, "text": truncate_text(row.text)} for row in batch.itertuples()],
        ensure_ascii=False,
        indent=2,
    )
    categories = sorted(descriptions.keys())
    return f"""
Classify each text into exactly one category using only these category descriptions.

Allowed categories: {json.dumps(categories, ensure_ascii=False)}
Category descriptions:
{description_block}

Texts to classify:
{text_block}

Return JSON exactly in this schema:
{{
  "predictions": [
    {{"text_id": "...", "predicted_category": "one allowed category", "confidence": 0.0, "rationale": "brief reason"}}
  ]
}}
The number of predictions must equal the number of input texts.
""".strip()


def classify_batch(
    client,
    batch: pd.DataFrame,
    description_records: dict[str, DescriptionRecord],
    batch_id: str,
    *,
    classification_model: str = CLASSIFICATION_MODEL,
) -> list[PredictionRecord]:
    prompt = classification_prompt(batch, description_records)
    parsed, response = gemini_generate_json(client, model=classification_model, prompt=prompt, temperature=0.0)
    raw_predictions = parsed.get("predictions", [])
    by_id = {str(item.get("text_id")): item for item in raw_predictions}
    allowed = set(description_records.keys())
    records: list[PredictionRecord] = []
    for row in batch.itertuples():
        pred = by_id.get(row.text_id, {})
        predicted = str(pred.get("predicted_category", "")).strip()
        if predicted not in allowed:
            predicted = "__INVALID__"
        confidence = pred.get("confidence")
        try:
            confidence = None if confidence is None else float(confidence)
        except Exception:
            confidence = None
        records.append({
            "text_id": row.text_id,
            "split": row.split,
            "true_category": row.category,
            "predicted_category": predicted,
            "confidence": confidence,
            "rationale": str(pred.get("rationale", "")),
            "model": classification_model,
            "batch_id": batch_id,
        })
    return records


def classify_split(
    client=None,
    split: str = EVAL_SPLIT,
    description_records: dict[str, DescriptionRecord] | None = None,
    *,
    output_prefix: str = "main",
    classification_model: str = CLASSIFICATION_MODEL,
) -> pd.DataFrame:
    client = client or get_genai_client()
    description_records = description_records or descriptions
    eval_df = df[df.split == split].sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    batches = [eval_df.iloc[i : i + CLASSIFICATION_BATCH_SIZE].copy() for i in range(0, len(eval_df), CLASSIFICATION_BATCH_SIZE)]
    all_records: list[PredictionRecord] = []
    for batch_idx, batch in enumerate(tqdm(batches, desc=f"Classifying {split} batches")):
        batch_id = stable_hash({
            "split": split,
            "batch_idx": batch_idx,
            "ids": batch.text_id.tolist(),
            "classification_model": classification_model,
            "description_hash": stable_hash(description_records),
            "output_prefix": output_prefix,
        })
        cache_path = paths.predictions / f"{output_prefix}_{split}_{batch_idx:05d}_{batch_id}.json"
        records = load_or_create_json(
            cache_path,
            lambda batch=batch, batch_id=batch_id: classify_batch(
                client,
                batch,
                description_records,
                batch_id,
                classification_model=classification_model,
            ),
        )
        all_records.extend(records)
    predictions_df = pd.DataFrame(all_records)
    merged = predictions_df.merge(df[["text_id", "text"]], on="text_id", how="left")
    output_path = paths.predictions / f"{output_prefix}_{split}_predictions.parquet"
    if not output_path.exists():
        merged.to_parquet(output_path, index=False)
    return merged


predictions = classify_split()
display(predictions.head())


## 8. Accuracy, F1, confusion matrix, and category tables

This section evaluates global accuracy and per-category recall/F1. The primary category-level accuracy is recall: among texts truly belonging to a category, the share correctly classified into that category.


In [ ]:
def compute_performance_tables(predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    y_true = predictions["true_category"].tolist()
    y_pred = predictions["predicted_category"].tolist()
    labels = sorted(df["category"].unique().tolist())
    global_metrics = pd.DataFrame([{
        "split": EVAL_SPLIT,
        "n": len(predictions),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
        "invalid_prediction_rate": float(np.mean(np.array(y_pred) == "__INVALID__")),
    }])
    report = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    category_metrics = (
        pd.DataFrame(report).T.reset_index(names="category")
        .query("category in @labels")
        .rename(columns={"recall": "category_accuracy"})
    )
    cm = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=labels, columns=labels)
    return global_metrics, category_metrics, cm


global_metrics, category_metrics, cm = compute_performance_tables(predictions)
global_metrics.to_csv(paths.metrics / f"{EVAL_SPLIT}_global_metrics.csv", index=False)
category_metrics.to_csv(paths.metrics / f"{EVAL_SPLIT}_category_metrics.csv", index=False)
cm.to_csv(paths.metrics / f"{EVAL_SPLIT}_confusion_matrix.csv")

display(global_metrics)
display(category_metrics.sort_values("category_accuracy", ascending=False))

plt.figure(figsize=(max(8, 0.6 * len(cm.columns)), max(6, 0.5 * len(cm.index))))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Confusion Matrix ({EVAL_SPLIT} split)")
plt.ylabel("True category")
plt.xlabel("Predicted category")
plt.tight_layout()
plt.savefig(paths.figures / f"{EVAL_SPLIT}_confusion_matrix.png", dpi=180)
plt.show()

plt.figure(figsize=(10, max(4, 0.35 * len(category_metrics))))
sns.barplot(data=category_metrics.sort_values("category_accuracy", ascending=True), x="category_accuracy", y="category", color="#4C78A8")
plt.title("Category Accuracy / Recall")
plt.xlim(0, 1)
plt.tight_layout()
plt.savefig(paths.figures / f"{EVAL_SPLIT}_category_accuracy.png", dpi=180)
plt.show()


## 9. Embeddings and semantic-distance preservation metric

### Metric definition

For each category, compute:

- `text_centroid`: centroid of embeddings for that category's training texts.
- `description_embedding`: embedding of the generated category description.
- `text_distance_matrix`: pairwise cosine distances between category text centroids.
- `description_distance_matrix`: pairwise cosine distances between category descriptions.

Then compute two preservation scores:

1. **Global distance preservation**: Spearman correlation between the upper triangles of the two distance matrices.
2. **Per-category preservation**: Spearman correlation between a category's row of text-centroid distances and its row of description distances. This measures whether each category's neighborhood relationships were preserved.

Higher values mean the descriptions preserve category geometry better.


In [ ]:
def embed_texts(client, texts: list[str], *, model: str = EMBEDDING_MODEL, task_type: str = "SEMANTIC_SIMILARITY") -> list[list[float]]:
    response = client.models.embed_content(
        model=model,
        contents=texts,
        config=genai_types.EmbedContentConfig(task_type=task_type) if genai_types is not None else None,
    )
    embeddings = []
    for emb in response.embeddings:
        values = getattr(emb, "values", None) or emb["values"]
        embeddings.append([float(x) for x in values])
    return embeddings


def cached_embeddings(client, key: str, texts: list[str]) -> np.ndarray:
    cache_path = paths.embeddings / f"{stable_hash({'key': key, 'model': EMBEDDING_MODEL, 'texts': texts})}.json"
    payload = load_or_create_json(cache_path, lambda: {"model": EMBEDDING_MODEL, "texts": texts, "embeddings": embed_texts(client, texts)})
    return np.array(payload["embeddings"], dtype=float)


def cosine_distance_matrix(vectors: np.ndarray, labels: list[str]) -> pd.DataFrame:
    mat = np.zeros((len(labels), len(labels)))
    for i in range(len(labels)):
        for j in range(len(labels)):
            mat[i, j] = 0.0 if i == j else cosine(vectors[i], vectors[j])
    return pd.DataFrame(mat, index=labels, columns=labels)


def safe_spearman(a: list[float] | np.ndarray, b: list[float] | np.ndarray) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if len(a) < 2 or np.nanstd(a) == 0 or np.nanstd(b) == 0:
        return np.nan
    return float(spearmanr(a, b).statistic)


def compute_distance_preservation(
    client=None,
    description_records: dict[str, DescriptionRecord] | None = None,
    *,
    output_prefix: str = "main",
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    client = client or get_genai_client()
    description_records = description_records or descriptions
    labels = sorted(df["category"].unique().tolist())

    text_centroids = []
    for category in tqdm(labels, desc="Embedding training texts by category"):
        texts = df[(df.category == category) & (df.split == "train")]["text"].map(truncate_text).tolist()
        embs = cached_embeddings(client, f"train_texts::{category}", texts)
        text_centroids.append(embs.mean(axis=0))
    text_centroids = np.vstack(text_centroids)

    description_texts = [description_records[c]["description"] for c in labels]
    description_vectors = cached_embeddings(client, f"{output_prefix}_descriptions", description_texts)

    text_distances = cosine_distance_matrix(text_centroids, labels)
    description_distances = cosine_distance_matrix(description_vectors, labels)

    upper = np.triu_indices(len(labels), k=1)
    global_preservation = safe_spearman(text_distances.values[upper], description_distances.values[upper])

    rows = []
    for category in labels:
        others = [c for c in labels if c != category]
        text_row = text_distances.loc[category, others].values
        desc_row = description_distances.loc[category, others].values
        preservation = safe_spearman(text_row, desc_row)
        mean_abs_distance_error = float(np.mean(np.abs(text_row - desc_row)))
        rows.append({
            "category": category,
            "distance_preservation": preservation,
            "mean_abs_distance_error": mean_abs_distance_error,
            "n_neighbors": len(others),
        })
    preservation_df = pd.DataFrame(rows)
    preservation_summary = pd.DataFrame([{
        "global_distance_preservation_spearman": global_preservation,
        "mean_category_distance_preservation": preservation_df["distance_preservation"].mean(skipna=True),
        "mean_abs_distance_error": preservation_df["mean_abs_distance_error"].mean(skipna=True),
    }])
    return preservation_summary, preservation_df, text_distances, description_distances


preservation_summary, preservation_df, text_distances, description_distances = compute_distance_preservation()
preservation_summary.to_csv(paths.metrics / "main_distance_preservation_summary.csv", index=False)
preservation_df.to_csv(paths.metrics / "main_category_distance_preservation.csv", index=False)
text_distances.to_csv(paths.metrics / "text_centroid_distance_matrix.csv")
description_distances.to_csv(paths.metrics / "main_description_distance_matrix.csv")

display(preservation_summary)
display(preservation_df.sort_values("distance_preservation", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(text_distances, cmap="viridis", ax=axes[0])
axes[0].set_title("Text-centroid cosine distances")
sns.heatmap(description_distances, cmap="viridis", ax=axes[1])
axes[1].set_title("Description cosine distances")
plt.tight_layout()
plt.savefig(paths.figures / "semantic_distance_matrices.png", dpi=180)
plt.show()


## 10. Core answer: is accuracy correlated with semantic-distance preservation?

This is the main analysis. It joins category accuracy with category-level distance preservation and tests correlation using Pearson and Spearman coefficients. It also bootstraps a confidence interval for Spearman correlation.

### Decision rule for the absolute answer

The notebook reports **YES** only when all of the following are true:

- Spearman correlation is positive.
- Spearman p-value is below 0.05.
- The 95% bootstrap confidence interval excludes 0.

Otherwise it reports **NO**. This conservative rule avoids claiming a relationship from noisy or underpowered category counts.


In [ ]:
def bootstrap_spearman_ci(x: np.ndarray, y: np.ndarray, *, n_boot: int = 5000, seed: int = RANDOM_SEED) -> tuple[float, float]:
    rng = np.random.default_rng(seed)
    vals = []
    n = len(x)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        val = safe_spearman(x[idx], y[idx])
        if not np.isnan(val):
            vals.append(val)
    if not vals:
        return (np.nan, np.nan)
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))


def summarize_accuracy_preservation_correlation(
    category_metrics_table: pd.DataFrame,
    preservation_table: pd.DataFrame,
    *,
    output_prefix: str = "main",
    make_plot: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Join accuracy and preservation tables, then produce the absolute YES/NO correlation answer."""
    analysis_table = category_metrics_table[["category", "category_accuracy", "f1-score", "support"]].merge(
        preservation_table, on="category", how="inner"
    )
    analysis_table = analysis_table.dropna(subset=["category_accuracy", "distance_preservation"])

    x = analysis_table["distance_preservation"].to_numpy(dtype=float)
    y = analysis_table["category_accuracy"].to_numpy(dtype=float)

    if len(analysis_table) >= 3 and np.nanstd(x) > 0 and np.nanstd(y) > 0:
        spearman = spearmanr(x, y)
        pearson = pearsonr(x, y)
        ci_low, ci_high = bootstrap_spearman_ci(x, y)
    else:
        spearman = type("Result", (), {"statistic": np.nan, "pvalue": np.nan})()
        pearson = type("Result", (), {"statistic": np.nan, "pvalue": np.nan})()
        ci_low, ci_high = np.nan, np.nan

    absolute_answer = (
        "YES"
        if (not np.isnan(spearman.statistic) and spearman.statistic > 0 and spearman.pvalue < 0.05 and ci_low > 0)
        else "NO"
    )

    summary = pd.DataFrame([{
        "experiment": output_prefix,
        "absolute_answer_accuracy_correlates_with_distance_preservation": absolute_answer,
        "n_categories": len(analysis_table),
        "spearman_rho": float(spearman.statistic) if not np.isnan(spearman.statistic) else np.nan,
        "spearman_p_value": float(spearman.pvalue) if not np.isnan(spearman.pvalue) else np.nan,
        "spearman_bootstrap_ci_low": ci_low,
        "spearman_bootstrap_ci_high": ci_high,
        "pearson_r": float(pearson.statistic) if not np.isnan(pearson.statistic) else np.nan,
        "pearson_p_value": float(pearson.pvalue) if not np.isnan(pearson.pvalue) else np.nan,
        "decision_rule": "YES iff Spearman > 0, p < 0.05, and bootstrap 95% CI excludes 0",
    }])

    analysis_table.to_csv(paths.metrics / f"{output_prefix}_accuracy_distance_preservation_by_category.csv", index=False)
    summary.to_csv(paths.metrics / f"{output_prefix}_accuracy_distance_preservation_correlation.csv", index=False)

    if make_plot:
        plt.figure(figsize=(9, 6))
        sns.regplot(data=analysis_table, x="distance_preservation", y="category_accuracy", ci=95, scatter_kws={"s": 80})
        for row in analysis_table.itertuples():
            plt.text(row.distance_preservation, row.category_accuracy, f" {row.category}", va="center", fontsize=8)
        plt.title(f"Accuracy vs. Semantic Distance Preservation ({output_prefix}): {absolute_answer}")
        plt.xlabel("Per-category distance preservation (Spearman)")
        plt.ylabel("Category accuracy / recall")
        plt.ylim(-0.05, 1.05)
        plt.tight_layout()
        plt.savefig(paths.figures / f"{output_prefix}_accuracy_vs_distance_preservation.png", dpi=180)
        plt.show()

    return summary, analysis_table


correlation_summary, analysis_df = summarize_accuracy_preservation_correlation(
    category_metrics, preservation_df, output_prefix="main"
)

display(correlation_summary)
display(analysis_df.sort_values("category_accuracy", ascending=False))

print(
    f"Absolute answer: {correlation_summary.loc[0, 'absolute_answer_accuracy_correlates_with_distance_preservation']}. "
    f"Spearman rho={correlation_summary.loc[0, 'spearman_rho']:.3f}, "
    f"p={correlation_summary.loc[0, 'spearman_p_value']:.4f}, "
    f"95% bootstrap CI=[{correlation_summary.loc[0, 'spearman_bootstrap_ci_low']:.3f}, "
    f"{correlation_summary.loc[0, 'spearman_bootstrap_ci_high']:.3f}]."
)


## 11. Optional ablation: Gemini 3.5 Flash descriptions in one request

This ablation tests a different description-generation strategy: all category descriptions are generated together in a single request using Gemini 3.5 Flash. Classification and metrics can then be re-run against the ablation descriptions.

Set `RUN_GEMINI_35_SINGLE_REQUEST_ABLATION=True` in the configuration cell to run it.


In [ ]:
def ablation_all_descriptions_prompt() -> str:
    categories_payload = []
    for category in sorted(df.category.unique()):
        train_examples = df[(df.category == category) & (df.split == "train")]["text"].tolist()
        random.Random(RANDOM_SEED).shuffle(train_examples)
        categories_payload.append({
            "category": category,
            "examples": [truncate_text(t) for t in train_examples[:MAX_TEXTS_PER_CATEGORY_FOR_DESCRIPTION]],
        })
    return f"""
You are building descriptions for every category in a text classifier.

For each category, write a precise reusable description with semantic boundaries, inclusion criteria, exclusion criteria, and edge cases.

Categories and training examples:
{json.dumps(categories_payload, ensure_ascii=False, indent=2)}

Return JSON exactly in this schema:
{{
  "descriptions": [
    {{"category": "...", "description": "..."}}
  ]
}}
Return one description for every category and no extra categories.
""".strip()


def run_ablation_descriptions(client=None) -> dict[str, DescriptionRecord]:
    client = client or get_genai_client()
    cache_path = paths.descriptions / f"ablation_all_descriptions_{ABLATION_DESCRIPTION_MODEL}.json"
    def create():
        prompt = ablation_all_descriptions_prompt()
        parsed, response = gemini_generate_json(client, model=ABLATION_DESCRIPTION_MODEL, prompt=prompt, temperature=0.2)
        now = datetime.now(timezone.utc).isoformat()
        records = {}
        for item in parsed.get("descriptions", []):
            category = str(item["category"]).strip()
            records[category] = {
                "category": category,
                "description": str(item["description"]).strip(),
                "model": ABLATION_DESCRIPTION_MODEL,
                "prompt_hash": stable_hash(prompt),
                "generated_at_utc": now,
                "raw_response": response_to_jsonable(response),
            }
        return records
    return load_or_create_json(cache_path, create)


if RUN_GEMINI_35_SINGLE_REQUEST_ABLATION:
    ablation_descriptions = run_ablation_descriptions()
    display(pd.DataFrame([{"category": k, "description": v["description"]} for k, v in ablation_descriptions.items()]))

    # Reuse the same classifier prompt and evaluation code, but isolate caches and metrics with an ablation prefix.
    ablation_predictions = classify_split(
        split=EVAL_SPLIT,
        description_records=ablation_descriptions,
        output_prefix="ablation_gemini35_single_request",
        classification_model=CLASSIFICATION_MODEL,
    )
    ablation_global_metrics, ablation_category_metrics, ablation_cm = compute_performance_tables(ablation_predictions)
    ablation_global_metrics.insert(0, "experiment", "ablation_gemini35_single_request")
    ablation_category_metrics.insert(0, "experiment", "ablation_gemini35_single_request")
    ablation_global_metrics.to_csv(paths.metrics / f"ablation_gemini35_single_request_{EVAL_SPLIT}_global_metrics.csv", index=False)
    ablation_category_metrics.to_csv(paths.metrics / f"ablation_gemini35_single_request_{EVAL_SPLIT}_category_metrics.csv", index=False)
    ablation_cm.to_csv(paths.metrics / f"ablation_gemini35_single_request_{EVAL_SPLIT}_confusion_matrix.csv")

    _, ablation_preservation_df, _, ablation_description_distances = compute_distance_preservation(
        description_records=ablation_descriptions,
        output_prefix="ablation_gemini35_single_request",
    )
    ablation_preservation_df.to_csv(paths.metrics / "ablation_gemini35_single_request_category_distance_preservation.csv", index=False)
    ablation_description_distances.to_csv(paths.metrics / "ablation_gemini35_single_request_description_distance_matrix.csv")

    ablation_correlation_summary, ablation_analysis_df = summarize_accuracy_preservation_correlation(
        ablation_category_metrics,
        ablation_preservation_df,
        output_prefix="ablation_gemini35_single_request",
    )

    comparison = pd.concat([
        global_metrics.assign(experiment="main_iterative_gemini31_flash_lite"),
        ablation_global_metrics,
    ], ignore_index=True, sort=False)
    correlation_comparison = pd.concat([
        correlation_summary.assign(experiment="main_iterative_gemini31_flash_lite"),
        ablation_correlation_summary,
    ], ignore_index=True, sort=False)

    display(comparison)
    display(correlation_comparison)
else:
    print("Ablation skipped. Set RUN_GEMINI_35_SINGLE_REQUEST_ABLATION=True to run.")


## 12. Experiment manifest

The manifest records enough configuration to reproduce the run and to make cached artifacts discoverable by future notebooks.


In [ ]:
manifest = {
    "experiment_id": EXPERIMENT_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_name": dataset.get("dataset_name"),
    "dataset_hash": stable_hash(dataset),
    "google_drive_json_path": GOOGLE_DRIVE_JSON_PATH,
    "models": {
        "description_model": DESCRIPTION_MODEL,
        "classification_model": CLASSIFICATION_MODEL,
        "ablation_description_model": ABLATION_DESCRIPTION_MODEL,
        "embedding_model": EMBEDDING_MODEL,
    },
    "parameters": {
        "random_seed": RANDOM_SEED,
        "train_size": TRAIN_SIZE,
        "eval_split": EVAL_SPLIT,
        "classification_batch_size": CLASSIFICATION_BATCH_SIZE,
        "max_texts_per_category_for_description": MAX_TEXTS_PER_CATEGORY_FOR_DESCRIPTION,
        "max_chars_per_text_in_prompts": MAX_CHARS_PER_TEXT_IN_PROMPTS,
        "reuse_cache": REUSE_CACHE,
    },
    "outputs": {
        "root": str(paths.root),
        "metrics": str(paths.metrics),
        "figures": str(paths.figures),
        "descriptions": str(paths.descriptions),
        "predictions": str(paths.predictions),
        "embeddings": str(paths.embeddings),
    },
}
manifest_path = paths.root / "manifest.json"
if not manifest_path.exists():
    write_json_once(manifest_path, manifest)
display(manifest)
